In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ConvNeXtTiny
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score
import matplotlib.pyplot as plt
import seaborn as sns

DATASET_PATH = '/kaggle/input/datasets/woltdwa/final-rgb/FINAL_RGB'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

ds_train = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH, 
    validation_split=0.2, 
    subset="training", 
    seed=123,
    image_size=IMG_SIZE, 
    batch_size=BATCH_SIZE
)

ds_val = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH, 
    validation_split=0.2, 
    subset="validation", 
    seed=123,
    image_size=IMG_SIZE, 
    batch_size=BATCH_SIZE
)

class_names = ds_train.class_names
AUTOTUNE = tf.data.AUTOTUNE

ds_train = ds_train.prefetch(buffer_size=AUTOTUNE)
ds_val = ds_val.prefetch(buffer_size=AUTOTUNE)

base_model = ConvNeXtTiny(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(256, activation='relu'),
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

model.fit(
    ds_train, 
    validation_data=ds_val, 
    epochs=5, 
    callbacks=[EarlyStopping(patience=2, restore_best_weights=True)]
)

base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5), 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

callbacks = [
    ModelCheckpoint('convnext_final.keras', save_best_only=True, monitor='val_accuracy'),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
]

history = model.fit(ds_train, validation_data=ds_val, epochs=10, callbacks=callbacks)

y_true, y_pred = [], []
for imgs, labels in ds_val:
    y_true.extend(labels.numpy())
    preds = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='macro')

print(f"\nResults -> Accuracy: {acc*100:.2f}%, Precision: {prec*100:.2f}%")

plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - ConvNeXt Tiny')
plt.show()